# Step 1 — Wind Scenario Construction (DK2)

We construct 20 wind production scenarios using historical capacity factor data.

## Methodology

### 1. Conversion to Power
Wind production is obtained as:

$$
W_{t,d} = CF_{t,d} \cdot P_{\max}
$$

where $P_{\max} = 500$ MW.

---

### 2. Scenario Selection (Median Bin Method)

- Compute daily energy:
$$
E_d = \sum_{t=1}^{24} W_{t,d}
$$

- Sort all days based on $E_d$
- Split into 20 bins
- Select the **median day** from each bin

This ensures:
- coverage of low → high wind regimes
- realistic intra-day profiles

---

### 3. Forecast Construction

For each selected scenario day $d$:

$$
\hat{W}_{t,d} = \frac{1}{k} \sum_{i=1}^{k} W_{t,d-i}
$$

with $k = 3$ previous days.

This creates a simple but realistic **forecast error structure**.

---

## Outputs

- **Realizations table**: 24 × 20 (hours × scenarios)
- **Forecast table**: 24 × 20
- Interactive plot:
  - Select scenario
  - Compare realization vs forecast


In [7]:
# ============================================
# STEP 1 — WIND SCENARIOS (DK2)
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


# ----------------------------
# PARAMETERS
# ----------------------------
P_max = 500  # MW
k = 3        # forecast window

# ----------------------------
# LOAD DATA
# ----------------------------
df = pd.read_csv(
    "raw/ninja-wind-country-DK-current_offshore-merra2.csv",
    skiprows=3
)

# Convert time
df["time"] = pd.to_datetime(df["time"])

# Filter DK2 (assuming already DK aggregate → we use as proxy)
df["date"] = df["time"].dt.date
df["hour"] = df["time"].dt.hour

# Pivot: rows = date, columns = hour
cf = df.pivot(index="date", columns="hour", values="DK02")

# Convert to power
wind_power = cf * P_max


# ----------------------------
# FILTER YEARS (IMPORTANT)
# ----------------------------

start_year = 2015
end_year = 2024

# Convert index to datetime (if not already)
wind_power.index = pd.to_datetime(wind_power.index)

# Filter
wind_power = wind_power[
    (wind_power.index.year >= start_year) &
    (wind_power.index.year <= end_year)
]


# ----------------------------
# DAILY ENERGY 
# ----------------------------
daily_energy = wind_power.sum(axis=1)

# Sort days
sorted_days = daily_energy.sort_values()

# ----------------------------
# MEDIAN BIN SELECTION
# ----------------------------
bins = np.array_split(sorted_days.index, 20)

selected_days = []
for b in bins:
    median_idx = len(b) // 2
    selected_days.append(b[median_idx])

# ----------------------------
# BUILD REALIZATION MATRIX
# ----------------------------
W = wind_power.loc[selected_days].values  # shape (20, 24)
W = W.T  # shape (24, 20)

# ----------------------------
# BUILD FORECAST MATRIX
# ----------------------------
W_hat = np.zeros_like(W)

all_days = wind_power.index.tolist()

for j, d in enumerate(selected_days):
    
    idx = all_days.index(d)
    
    # Ensure enough past days
    if idx >= k:
        prev_days = all_days[idx-k:idx]
        W_hat[:, j] = wind_power.loc[prev_days].mean(axis=0).values
    else:
        # fallback: use same day (rare edge)
        W_hat[:, j] = wind_power.loc[d].values

# ----------------------------
# CREATE TABLES (FINAL CLEAN)
# ----------------------------

hours = np.arange(24)

scenario_names = [
    f"Scenario_{i+1} ({selected_days[i]})"
    for i in range(len(selected_days))
]

# Build WITHOUT index
realizations_df = pd.DataFrame(W, columns=scenario_names)
realizations_df.insert(0, "Hour", hours)

forecast_df = pd.DataFrame(W_hat, columns=scenario_names)
forecast_df.insert(0, "Hour", hours)



print(realizations_df.head())
print(forecast_df.head())

print("Realizations shape:", realizations_df.shape)
print("Forecast shape:", forecast_df.shape)



   Hour  Scenario_1 (2023-11-12 00:00:00)  Scenario_2 (2016-04-24 00:00:00)  \
0     0                            7.6405                          100.9235   
1     1                            8.0015                           94.8755   
2     2                            9.3600                           86.9055   
3     3                           10.9945                           79.0275   
4     4                           12.0350                           68.5340   

   Scenario_3 (2018-04-28 00:00:00)  Scenario_4 (2023-04-09 00:00:00)  \
0                           85.9795                           43.8110   
1                           86.3500                           48.3050   
2                           69.8230                           52.2715   
3                           42.7830                           52.0155   
4                           20.9000                           50.5830   

   Scenario_5 (2024-09-22 00:00:00)  Scenario_6 (2015-06-04 00:00:00)  \
0            

In [8]:
# ----------------------------
# SAVE OUTPUTS
# ----------------------------

import os

# Ensure folder exists
os.makedirs("processed", exist_ok=True)

# Save CSVs
realizations_df.to_csv("processed/wind_realizations.csv",
    index=False)
forecast_df.to_csv("processed/wind_forecasts.csv",
    index=False)


print("Files saved in data/processed/")

Files saved in data/processed/


In [9]:
from ipywidgets import interact

# ----------------------------
# INTERACTIVE PLOT
# ----------------------------
def plot_scenario(scenario=1):
    plt.figure(figsize=(10,5))
    
    plt.plot(hours, realizations_df[f"Scenario_{scenario}"], 
             label="Realization", linewidth=2)
    
    plt.plot(hours, forecast_df[f"Scenario_{scenario}"], 
             linestyle="--", label="Forecast", linewidth=2)
    
    plt.xlabel("Hour")
    plt.ylabel("Wind Power [MW]")
    plt.title(f"Scenario {scenario}")
    plt.legend()
    plt.grid()
    plt.show()

interact(plot_scenario, scenario=(1,20))

<Figure size 1000x500 with 0 Axes>

interactive(children=(IntSlider(value=1, description='scenario', max=20, min=1), Output()), _dom_classes=('wid…

<function __main__.plot_scenario(scenario=1)>